In [1]:
# ════════════════════════════════════════════
# CELLULE 1 — Imports & setup
# ════════════════════════════════════════════
import sys
sys.path.append(r"C:\Users\user\Desktop\nba_copilot")

import numpy as np
import pandas as pd
import statsmodels.formula.api as smf
import statsmodels.api as sm
from sklearn.metrics import mean_absolute_error
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder

from Composant3.Valorisation.config import SALARY_CAP_REF
from Composant3.Valorisation.data_loader import load_pairs
from Composant3.Valorisation.feature_engineering import (
    build_features,
    NUM_BASE, CAT_FEATURES,
    # FE individuelles pour tests à la carte
    add_rarete_par_poste,
    add_interactions_position_skill,
    add_polyvalence,
    add_age_peak_interaction,
    add_scoring_efficiency,
    add_clean_playmaking,
    add_bankability_score,
    add_archetype_kmeans,
)
from Composant3.Valorisation.model import BetaValuationModel

In [2]:
# ════════════════════════════════════════════
# CELLULE 2 — Chargement des données
# ════════════════════════════════════════════
pairs = load_pairs(segment="middle_class")
print(f"\nDataset prêt : {len(pairs)} paires")
print(f"Colonnes : {pairs.shape[1]}")

[data_loader] segment='middle_class'  →  601 paires
  Par saison :
saison
2021-22    143
2022-23    147
2023-24    159
2024-25    152
  cap_hit_pct  mean=0.1169  std=0.0532  min=0.0501  max=0.2500

Dataset prêt : 601 paires
Colonnes : 60


In [3]:
# ════════════════════════════════════════════
# CELLULE 3 — Baseline et reproductibilité
# Reproduit le MAE de référence : Beta 44f → ~4.01M$
# ════════════════════════════════════════════
def run_beta(df, num_features, cat_features, label=""):
    """Helper : entraîne Beta Regression et retourne MAE en $."""
    model = BetaValuationModel()
    model.fit(df, num_features, cat_features)
    metrics = model.evaluate(df)
    print(f"{label:<40} MAE={metrics['mae_pct']:.4f} → {metrics['mae_dollars']:>12,.0f}$  "
          f"({metrics['gain_vs_baseline']:+.1%} vs baseline)")
    return model, metrics


# Beta Regression 44f sans FE — doit donner ~4.01M$
model_base, m_base = run_beta(pairs, NUM_BASE, CAT_FEATURES,
                               label="Beta 44f (base, sans FE)")

C:\Users\user\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


Beta 44f (base, sans FE)                 MAE=0.0288 →    4,046,142$  (+35.7% vs baseline)


In [4]:
# ════════════════════════════════════════════
# CELLULE 4 — FE validées (état actuel : MAE ~3.95M$)
# ════════════════════════════════════════════
pairs_v3, num_v3, cat_v3 = build_features(
    pairs,
    use_rarete=True,
    use_interactions=True,
    use_polyvalence=True,
)

model_v3, m_v3 = run_beta(pairs_v3, num_v3, cat_v3,
                           label="Beta V3 (rareté+interact+poly)")

[feature_engineering] 58 features numériques  + 2 catégorielles
Beta V3 (rareté+interact+poly)           MAE=0.0280 →    3,935,307$  (+37.5% vs baseline)


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


In [5]:
# ════════════════════════════════════════════
# CELLULE 5 — Test des FE restantes 🔴
# Chaque FE testée séparément, base = V3
# ════════════════════════════════════════════
print("=" * 65)
print("CONTRIBUTION DES FE RESTANTES (ajoutées sur la base V3)")
print("=" * 65)

candidates = {
    "age_peak_interaction": dict(use_rarete=True, use_interactions=True,
                                  use_polyvalence=True, use_age_peak=True),
    "scoring_efficiency":   dict(use_rarete=True, use_interactions=True,
                                  use_polyvalence=True, use_scoring_eff=True),
    "clean_playmaking":     dict(use_rarete=True, use_interactions=True,
                                  use_polyvalence=True, use_clean_pm=True),
    "bankability_score":    dict(use_rarete=True, use_interactions=True,
                                  use_polyvalence=True, use_bankability=True),
    "archetype_kmeans":     dict(use_rarete=True, use_interactions=True,
                                  use_polyvalence=True, use_archetype=True),
}

resultats = {"Beta V3 (base)": m_v3["mae_dollars"]}

for nom, kwargs in candidates.items():
    df_test, num_test, cat_test = build_features(pairs, **kwargs)
    _, m = run_beta(df_test, num_test, cat_test, label=f"+ {nom}")
    resultats[f"+ {nom}"] = m["mae_dollars"]

print("\n── Récapitulatif ──")
for nom, mae_usd in sorted(resultats.items(), key=lambda x: x[1]):
    delta = mae_usd - resultats["Beta V3 (base)"]
    signe = "✅" if delta < 0 else "❌"
    print(f"  {signe}  {nom:<35}  {mae_usd:>12,.0f}$  ({delta:+,.0f}$)")

CONTRIBUTION DES FE RESTANTES (ajoutées sur la base V3)
[feature_engineering] 59 features numériques  + 2 catégorielles


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


+ age_peak_interaction                   MAE=0.0268 →    3,774,195$  (+40.0% vs baseline)
[feature_engineering] 59 features numériques  + 2 catégorielles


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


+ scoring_efficiency                     MAE=0.0280 →    3,934,641$  (+37.5% vs baseline)
[feature_engineering] 59 features numériques  + 2 catégorielles


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


+ clean_playmaking                       MAE=0.0280 →    3,933,354$  (+37.5% vs baseline)
[feature_engineering] 59 features numériques  + 2 catégorielles


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


+ bankability_score                      MAE=0.0280 →    3,930,511$  (+37.5% vs baseline)
[feature_engineering] 64 features numériques  + 2 catégorielles
+ archetype_kmeans                       MAE=0.0279 →    3,921,708$  (+37.7% vs baseline)

── Récapitulatif ──
  ✅  + age_peak_interaction                  3,774,195$  (-161,112$)
  ✅  + archetype_kmeans                      3,921,708$  (-13,599$)
  ✅  + bankability_score                     3,930,511$  (-4,796$)
  ✅  + clean_playmaking                      3,933,354$  (-1,953$)
  ✅  + scoring_efficiency                    3,934,641$  (-666$)
  ❌  Beta V3 (base)                          3,935,307$  (+0$)


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


In [6]:
# ════════════════════════════════════════════
# CELLULE 6 — Meilleure combinaison
# Après avoir lu les résultats de la cellule 5,
# activer les FE qui améliorent et relancer ici.
# ════════════════════════════════════════════

# Modifier les flags selon les résultats observés en cellule 5
pairs_best, num_best, cat_best = build_features(
    pairs,
    use_rarete=True,
    use_interactions=True,
    use_polyvalence=True,
    use_age_peak=True,      # ← à activer si cellule 5 montre un gain
    use_scoring_eff=True,   # ← idem
    use_clean_pm=True,
    use_bankability=True,
    use_archetype=True,
)

model_best, m_best = run_beta(pairs_best, num_best, cat_best,
                               label="Beta BEST (combinaison finale)")

print(f"\n{'='*50}")
print(f"Récapitulatif final")
print(f"{'='*50}")
print(f"Baseline naïve    :  {m_base['baseline_mae_dollars']:>12,.0f}$")
print(f"Beta 44f (base)   :  {m_base['mae_dollars']:>12,.0f}$  "
      f"({m_base['gain_vs_baseline']:+.1%})")
print(f"Beta V3           :  {m_v3['mae_dollars']:>12,.0f}$  "
      f"({m_v3['gain_vs_baseline']:+.1%})")
print(f"Beta BEST         :  {m_best['mae_dollars']:>12,.0f}$  "
      f"({m_best['gain_vs_baseline']:+.1%})")

[feature_engineering] 68 features numériques  + 2 catégorielles
Beta BEST (combinaison finale)           MAE=0.0268 →    3,768,558$  (+40.1% vs baseline)

Récapitulatif final
Baseline naïve    :     6,292,865$
Beta 44f (base)   :     4,046,142$  (+35.7%)
Beta V3           :     3,935,307$  (+37.5%)
Beta BEST         :     3,768,558$  (+40.1%)


C:\Users\user\AppData\Roaming\Python\Python313\site-packages\statsmodels\genmod\generalized_linear_model.py:1486: SingularMatrixWarning: The design matrix is rank-deficient. The model parameters are not uniquely determined.
  wls_results = wls_model.fit(method=wls_method2)


In [ ]:
# ════════════════════════════════════════════
# CELLULE 7 — Test sur un joueur
# Proto de get_player_value()
# ════════════════════════════════════════════
from sqlalchemy import create_engine, text
from Composant3.Valorisation.config import DB_URL, FEATURE_VERSION, SALARY_CAP

PLAYER_ID   = 1629673   # Jordan Poole
SAISON      = "2024-25"
CAP_SAISON  = SALARY_CAP["2024-25"]

engine = create_engine(DB_URL)
with engine.connect() as conn:
    # On charge toute la saison pour que les FE transductives (rank, kmeans) fonctionnent !
    season_df = pd.read_sql(text("""
        SELECT pf.*, p.position, p.draft_year, p.draft_round, p.draft_number
        FROM player_features pf
        LEFT JOIN players p ON pf.player_id = p.player_id
        WHERE pf.saison = :saison
        AND pf.feature_version = :v
    """), conn, params={"saison": SAISON, "v": FEATURE_VERSION})

    cap_hit = pd.read_sql(text("""
        SELECT cap_hit FROM contracts
        WHERE player_id = :pid AND saison = :saison
    """), conn, params={"pid": PLAYER_ID, "saison": SAISON})

# Appliquer les mêmes FE que model_best sur toute la saison
season_fe, _, _ = build_features(season_df,
                                   use_rarete=True,
                                   use_interactions=True,
                                   use_polyvalence=True,
                                   use_age_peak=True,
                                   use_scoring_eff=True,
                                   use_clean_pm=True,
                                   use_bankability=True,
                                   use_archetype=True)

# Isoler le joueur
player_fe = season_fe[season_fe["player_id"] == PLAYER_ID].copy()

result = model_best.predict_surplus(
    player_fe,
    cap_hit=float(cap_hit["cap_hit"].iloc[0]),
    salary_cap=CAP_SAISON,
)

print(f"\n{'='*50}")
print(f"VALORISATION — Jordan Poole ({SAISON})")
print(f"{'='*50}")
print(f"Valeur estimée  (P50) : {result['valeur_dollars']:>12,.0f}$  "
      f"({result['valeur_pct']*100:.1f}% du cap)")
print(f"Fourchette 80%        : {result['p10_dollars']:>12,.0f}$ "
      f"— {result['p90_dollars']:,.0f}$")
print(f"Salaire actuel        : {result['cap_hit']:>12,.0f}$")
print(f"Surplus value         : {result['surplus']:>+12,.0f}$")
print(f"Signal                : {result['signal']}")
\

ImportError: cannot import name 'SALARY_CAP' from 'Composant3.config' (C:\Users\user\Desktop\nba_copilot\Composant3\config.py)